In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # put this at the very top of your script


from dataclasses import dataclass
import torch
import torch.nn.functional as F

from tools_llada import ConfKSorter, ConfCollectorInterface, BlockDiffusionQuotaHelper
from plugins_llada import InspectorPlugin


class SimpleLogitsSnapshot:

    def _regularize(self, sample, target):
        return  sample[:, :target.shape[1]]
    # end

    def __init__(self, logits, x, y, id_mask):
        self.id_mask = id_mask

        self.logits = logits

        self.x = self._regularize(x, logits)
        self.y = self._regularize(y, logits)

        self.x0 = torch.argmax(self.logits, dim=-1)

        self.p_finalized = torch.zeros(self.x.shape, dtype=torch.float64).to(self.x.device)
    # end

    def transform_logits(self, collector):

        logits_tranform = self.logits
        p = F.softmax(logits_tranform.to(torch.float64), dim=-1)

        index_p_all = collector.get_index(self)
        print(f'JINYUJ: index_p_all: {index_p_all.shape} for p: {p.shape}')


        x0_p = torch.gather(p, dim=-1, index=index_p_all).squeeze(-1)
        print(f'JINYUJ: x0_p: {x0_p.shape} {x0_p.dtype}, {torch.finfo(x0_p.dtype).min}')
        
        neg_inf = torch.tensor(torch.finfo(x0_p.dtype).min, device=x0_p.device, dtype=x0_p.dtype)

        mask_mask = self.x == self.id_mask
        conf = torch.where(mask_mask, x0_p, neg_inf)  # (B, L)   # so only the masked part has confidence

        return conf
    # end

    def materilize_by_idx_(self, idx, conf):
        print(f'self.x0: {self.x0.shape}, idx: {idx.shape}, conf: {conf.shape}')
        
        x0 = torch.gather(self.x0, dim=-1, index=idx)
        conf = torch.gather(conf, dim=-1, index=idx)
        print(f'after gather x0: {x0.shape}, conf: {conf.shape}, self.p_finalized: {self.p_finalized.shape}')
        self.x.scatter_(1, idx, x0)
        self.p_finalized.scatter_(1, idx, conf)
    # end


    #TODO: this is for next round
    def update_logits_(self, idx_logits, logits):
        logits_update = torch.gather(logits, dim=, index=idx_logits)

        self.logits.scatter_(1, logits, idx_logits)
        x0 = torch.argmax(logits, dim=-1)
        self.x0.scatter_(1, x0, idx_logits)
    # end
# end



In [2]:
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

from tools_llada import TopKSorter, TruthCollector
from plugins_llada import SaveKVPreviousPlugin_Disabled, CachePastKVPlugin_Disabled
from datasets import load_dataset, Dataset

from dataset_llada import LIST_DATASET
from datapreprocess_llada import parse_lines_with_index, merge_subdocs, PATTEN_REG_WIKI
from dataprocess_llada import Tokenizer_wiki_simple, Collater_wiki_simple

from modeling_yukai_llada import LLaDAModelLM

@dataclass
class DiffusionConfig:
    id_model: str
    len_prompt: int
    len_target: int    
    num_blocks: int
    num_unmask_per_step: int
    step_per_block: int
    size_block: int
    id_mask: int
    size_batch: int
    device: str
    klass_sorter: ConfKSorter
    klass_collector: ConfCollectorInterface
    klass_save_kv_previous: InspectorPlugin
    klass_cache_past_kv: InspectorPlugin
# end



config = DiffusionConfig(
    id_model='GSAI-ML/LLaDA-8B-Base',
    len_prompt=16,
    len_target=64,
    num_blocks=2,
    num_unmask_per_step=1,
    step_per_block=32,
    size_block=32,
    id_mask=126336,
    size_batch=1,
    device='cuda:0',
    klass_sorter=TopKSorter,
    klass_collector=TruthCollector,
    klass_save_kv_previous=SaveKVPreviousPlugin_Disabled,
    klass_cache_past_kv=CachePastKVPlugin_Disabled
)


'''load dataset first'''
name_dataset = LIST_DATASET[1]
ds = load_dataset(*name_dataset, split='all')
docs, _ = parse_lines_with_index(PATTEN_REG_WIKI, ds['text'])
docs = docs['subdocs']

samples = []
for doc in docs:
    lines_1 = doc['texts']
    paragraph_1 = ' '.join(lines_1)
    lines_remain, titles = merge_subdocs(doc['subdocs'])
    paragraph_remain = ' '.join(lines_remain)
    prefix = paragraph_1
    target = paragraph_remain
    samples.append({'text': paragraph_1 + ' ' + paragraph_remain})
# end

ds_origin = Dataset.from_list(samples[:3])


'''initialize constant hyper-parameters'''

'''load tokenizer'''
tokenizer = AutoTokenizer.from_pretrained(
    config.id_model,
    trust_remote_code=True
)

if tokenizer.padding_side != 'left':
    tokenizer.padding_side = 'left'
# end
assert tokenizer.pad_token_id != 126336




'''load model'''
model_kwargs = {}
model = LLaDAModelLM.from_pretrained(
    config.id_model,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    **model_kwargs
)

model = model.eval().to(config.device)


'''start to handle dataset based on hyper-parameter'''
len_max = config.len_prompt + config.len_target
ds = ds_origin\
    .filter(lambda x: x["text"] is not None and len(x["text"].strip()) > 0)\
    .map(Tokenizer_wiki_simple(tokenizer, len_max), remove_columns=ds_origin.column_names)\
    .filter(lambda x: x["length"] >= len_max)\
    .sort("length")
# end

'''prepare dataloader'''
loader = DataLoader(
    ds,
    batch_size=config.size_batch,
    shuffle=False,                 # keep sorted order
    collate_fn=Collater_wiki_simple(len_max, config.len_prompt, config.len_target, config.id_mask),
    drop_last=False
)

/home/exx/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
Filter: 100%|██████████| 3/3 [00:00<00:00, 3082.54 examples/s]


In [3]:
# 处理past_key_values
def run_model_semi(model, x, y, config_diffusion):

    num_blocks = config_diffusion.num_blocks
    step_per_block = config_diffusion.step_per_block
    size_block = config_diffusion.size_block
    id_mask = config_diffusion.id_mask
    len_prompt = config_diffusion.len_prompt
    sorter = config_diffusion.klass_sorter()
    collector = config_diffusion.klass_collector()
    
    # p_final = torch.zeros(x.shape, dtype=torch.float64, device=x.device)
    position_start = 0

    for id_block in range(1, num_blocks+1):
        position_end = position_start + id_block * size_block
        mask_mask_blk = x[:,:position_end] == id_mask

        B = x.shape[0]
        L = position_end - position_start
        
        # idx_denoising = torch.arange(position_start, position_end, dtype=torch.long).unsqueeze(0).expand(B, L).to(x.device)
        idx_denoising = torch.arange(position_start, position_end, dtype=torch.long).to(x.device)
        quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)

        for step in range(step_per_block):
            # logits = model(torch.gather(x, -1, idx_denoising), idx_current=idx_denoising).logits
            x_denoising,  y_denoising= x[:, idx_denoising], y[:, idx_denoising]
            logits = model(x_denoising, idx_current=idx_denoising).logits

            snapshot = SimpleLogitsSnapshot(logits, x_denoising, y_denoising, id_mask)
            conf_snapshot = snapshot.transform_logits(collector)
            print('JINYUJ: conf_snapshot : {}'.format(conf_snapshot.shape))

            idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)
            num_unmask = quota_helper.get_quota(step)
            print('JINYUJ: num_unmask: {}, idx_sorted_by_conf: {}'.format(num_unmask, idx_sorted_by_conf.shape))

            idx_transform = idx_sorted_by_conf[:, :num_unmask]
            print('JINYUJ: idx_transform: {}'.format(idx_transform.shape))

            
            snapshot.materilize_by_idx_(idx_transform, conf_snapshot)
            # p_final.scatter_(1, idx_transform, snapshot.get_p_finalized())
        # end for step

    # end for block

    # return torch.gather(p_final, dim=-1, index=torch.arange(len_prompt, x.shape[-1]))
# end

In [4]:
from tqdm import tqdm

model.fill_plugin(config.klass_cache_past_kv).fill_plugin(config.klass_save_kv_previous)

'''start the evaluation process'''
for id_batch, batch in enumerate(tqdm(loader)):
    run_model_semi(
        model,
        batch['ids_prompt_masked_full'].to(config.device),
        batch['ids_target_masked_full'].to(config.device),
        config
    )

    break
# end for

  0%|          | 0/3 [00:00<?, ?it/s]

JINYUJ: index_p_all: torch.Size([1, 32, 1]) for p: torch.Size([1, 32, 126464])
JINYUJ: x0_p: torch.Size([1, 32]) torch.float64, -1.7976931348623157e+308
JINYUJ: conf_snapshot : torch.Size([1, 32])
[DEBUG][argsort:70] confidence: torch.Size([1, 32]) [FILE] /home/exx/Workspace/yukai_llada_base/yukai_llada_new/tools_llada.py
[DEBUG][argsort:48] idx_sorted: torch.Size([1, 32]) [FILE] /home/exx/Workspace/yukai_llada_base/yukai_llada_new/tools_llada.py
JINYUJ: num_unmask: tensor([1], device='cuda:0'), idx_sorted_by_conf: torch.Size([1, 32])
JINYUJ: idx_transform: torch.Size([1, 1])
self.x0: torch.Size([1, 32]), idx: torch.Size([1, 1]), conf: torch.Size([1, 32])
after gather x0: torch.Size([1, 1]), conf: torch.Size([1, 1]), self.p_finalized: torch.Size([1, 32])
JINYUJ: index_p_all: torch.Size([1, 32, 1]) for p: torch.Size([1, 32, 126464])
JINYUJ: x0_p: torch.Size([1, 32]) torch.float64, -1.7976931348623157e+308
JINYUJ: conf_snapshot : torch.Size([1, 32])
[DEBUG][argsort:70] confidence: torch.

  0%|          | 0/3 [00:03<?, ?it/s]

JINYUJ: index_p_all: torch.Size([1, 64, 1]) for p: torch.Size([1, 64, 126464])
JINYUJ: x0_p: torch.Size([1, 64]) torch.float64, -1.7976931348623157e+308
JINYUJ: conf_snapshot : torch.Size([1, 64])
[DEBUG][argsort:70] confidence: torch.Size([1, 64]) [FILE] /home/exx/Workspace/yukai_llada_base/yukai_llada_new/tools_llada.py
[DEBUG][argsort:48] idx_sorted: torch.Size([1, 64]) [FILE] /home/exx/Workspace/yukai_llada_base/yukai_llada_new/tools_llada.py
JINYUJ: num_unmask: tensor([1], device='cuda:0'), idx_sorted_by_conf: torch.Size([1, 64])
JINYUJ: idx_transform: torch.Size([1, 1])
self.x0: torch.Size([1, 64]), idx: torch.Size([1, 1]), conf: torch.Size([1, 64])
after gather x0: torch.Size([1, 1]), conf: torch.Size([1, 1]), self.p_finalized: torch.Size([1, 64])
JINYUJ: index_p_all: torch.Size([1, 64, 1]) for p: torch.Size([1, 64, 126464])
JINYUJ: x0_p: torch.Size([1, 64]) torch.float64, -1.7976931348623157e+308
JINYUJ: conf_snapshot : torch.Size([1, 64])
[DEBUG][argsort:70] confidence: torch.